Camera testing using openCv

In [ ]:
import cv2

# Open webcam
cap = cv2.VideoCapture(1)

# Check camera
if not cap.isOpened():
    print("Cannot open camera")
    exit()

while True:
    ret, frame = cap.read()

    # Check frame
    if not ret or frame is None:
        print("Failed to grab frame")
        break

    # Convert to gray
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Show frame
    cv2.imshow("Webcam", gray)

    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Google colab cannot directly access camera, so using javascript to capture image for face detection


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

# Load Haar Cascade
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# JavaScript to capture image
def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const capture = document.createElement('button');
          capture.textContent = 'Capture';
          div.appendChild(capture);

          const video = document.createElement('video');
          video.style.display = 'block';

          const stream = await navigator.mediaDevices.getUserMedia({video: true});

          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();

          google.colab.output.setIframeHeight(document.body.scrollHeight, true);

          await new Promise((resolve) => capture.onclick = resolve);

          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);

          stream.getVideoTracks()[0].stop();
          div.remove();

          return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(js)

    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

# Take photo
image_path = take_photo()

# Read image
img = cv2.imread(image_path)

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Detect faces
faces = face_cascade.detectMultiScale(gray, 1.1, 5)

# Draw rectangles
for (x, y, w, h) in faces:
    cv2.rectangle(img, (x, y),
                  (x+w, y+h),
                  (255, 0, 0), 2)

# Show image
from google.colab.patches import cv2_imshow
cv2_imshow(img)

In [ ]:
# Import libraries
import cv2
import numpy as np
import dlib

In [ ]:
# (0) in VideoCapture is used to
# connect to your computer's default camera
cap = cv2.VideoCapture(1)
# Get the coordinates
detector = dlib.get_frontal_face_detector()

Face counter after capturing picture

In [ ]:

import cv2
import dlib
from google.colab.patches import cv2_imshow
from IPython.display import Javascript, display
from google.colab.output import eval_js
import numpy as np
from base64 import b64decode

# JavaScript function to capture webcam image
def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.body.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;

      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getVideoTracks()[0].stop();
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')

    display(js)

    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

# Load dlib face detector
detector = dlib.get_frontal_face_detector()

while True:

    # Capture image from webcam
    filename = take_photo()

    # Read image
    frame = cv2.imread(filename)

    # Flip image
    frame = cv2.flip(frame, 1)

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = detector(gray)

    # Face counter
    i = 0

    for face in faces:

        x, y = face.left(), face.top()
        x1, y1 = face.right(), face.bottom()

        # Draw rectangle around face
        cv2.rectangle(frame, (x, y), (x1, y1), (0, 255, 0), 2)

        i += 1

        # Display face number
        cv2.putText(frame,
                    'Face num ' + str(i),
                    (x - 10, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2)

        print("Face detected:", i)

    # Show image
    cv2_imshow(frame)

    # Ask user whether to continue
    cont = input("Press Enter to continue or type q to quit: ")

    if cont.lower() == 'q':
        break

In [ ]:


import cv2
import dlib
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io

# Load face detector
detector = dlib.get_frontal_face_detector()

# JavaScript webcam stream
display(Javascript('''
async function startVideo() {
    const video = document.createElement('video');
    video.setAttribute('playsinline', '');
    video.style.display = 'block';
    document.body.appendChild(video);

    const stream = await navigator.mediaDevices.getUserMedia({video: true});
    video.srcObject = stream;

    await video.play();

    window.video = video;
}

async function captureFrame() {
    const canvas = document.createElement('canvas');
    canvas.width = window.video.videoWidth;
    canvas.height = window.video.videoHeight;

    canvas.getContext('2d').drawImage(window.video, 0, 0);

    return canvas.toDataURL('image/jpeg', 0.8);
}

startVideo();
'''))

# Continuous loop
while True:

    # Capture frame from browser webcam
    data = eval_js('captureFrame()')

    # Decode image
    binary = b64decode(data.split(',')[1])
    image = Image.open(io.BytesIO(binary))

    # Convert to OpenCV format
    frame = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

    # Flip frame
    frame = cv2.flip(frame, 1)

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = detector(gray)

    # Face counter
    i = 0

    for face in faces:

        x, y = face.left(), face.top()
        x1, y1 = face.right(), face.bottom()

        # Draw rectangle
        cv2.rectangle(frame, (x, y), (x1, y1), (0, 255, 0), 2)

        # Increment counter
        i += 1

        # Display face number
        cv2.putText(frame,
                    'Face num ' + str(i),
                    (x - 10, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2)

    # Show frame
    cv2_imshow(frame)

    # Press Ctrl+C in Colab to stop

In [ ]:
import cv2

cap = cv2.VideoCapture(0, cv2.CAP_MSMF)

print("Camera opened:", cap.isOpened())

ret, frame = cap.read()

print("Frame captured:", ret)

cap.release()

In [ ]:
!pip install ultralytics
!pip install opencv-python


In [ ]:
!wget -O yolov8n-face.pt \
https://github.com/akanametov/yolo-face/releases/yolov8n-face.pt

In [ ]:
# Remove corrupted file
!rm -f yolov8n-face.pt

In [ ]:
!wget -O yolov11n-face.pt \
https://github.com/akanametov/yolo-face/releases/download/v0.0.0/yolov11n-face.pt

In [ ]:
!pip install huggingface_hub

Face detection using YOLO algorithm

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

from base64 import b64decode
import cv2
import numpy as np
import time

# Load model
model = YOLO("yolov8n.pt")


# Create webcam ONCE
display(Javascript('''
async function initCamera() {

    if (window.localStream) {
        return;
    }

    const div = document.createElement('div');
    div.id = 'camera_container';

    const video = document.createElement('video');
    video.id = 'camera_video';

    video.style.display = 'block';
    video.width = 400;

    document.body.appendChild(div);
    div.appendChild(video);

    window.localStream =
        await navigator.mediaDevices.getUserMedia({
            video: true
        });

    video.srcObject = window.localStream;

    await video.play();
}

async function captureFrame() {

    const video =
        document.getElementById('camera_video');

    const canvas =
        document.createElement('canvas');

    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    canvas
        .getContext('2d')
        .drawImage(video, 0, 0);

    return canvas.toDataURL(
        'image/jpeg',
        0.8
    );
}

async function stopCamera() {

    if (window.localStream) {

        window.localStream
            .getTracks()
            .forEach(track => track.stop());

        window.localStream = null;
    }

    const video =
        document.getElementById('camera_video');

    if (video) {

        video.pause();
        video.srcObject = null;
    }

    const container =
        document.getElementById('camera_container');

    if (container) {
        container.remove();
    }

    return "Camera stopped";
}
'''))

# Start camera
eval_js('initCamera()')

try:

    for _ in range(50):

        data = eval_js('captureFrame()')

        binary = b64decode(
            data.split(',')[1]
        )

        jpg = np.frombuffer(
            binary,
            dtype=np.uint8
        )

        frame = cv2.imdecode(
            jpg,
            cv2.IMREAD_COLOR
        )

        results = model(frame)

        annotated_frame = results[0].plot()

        cv2_imshow(annotated_frame)

        time.sleep(0.1)

finally:

    print(eval_js('stopCamera()'))

In [ ]:
!pip install deepface

In [ ]:
from deepface import DeepFace
from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

from base64 import b64decode
import cv2
import numpy as np
import time

# Create webcam ONCE
display(Javascript('''
async function initCamera() {

    if (window.localStream) {
        return;
    }

    const div = document.createElement('div');
    div.id = 'camera_container';

    const video = document.createElement('video');
    video.id = 'camera_video';

    video.style.display = 'block';
    video.width = 400;

    document.body.appendChild(div);
    div.appendChild(video);

    window.localStream =
        await navigator.mediaDevices.getUserMedia({
            video: true
        });

    video.srcObject = window.localStream;

    await video.play();
}

async function captureFrame() {

    const video =
        document.getElementById('camera_video');

    const canvas =
        document.createElement('canvas');

    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    canvas
        .getContext('2d')
        .drawImage(video, 0, 0);

    return canvas.toDataURL(
        'image/jpeg',
        0.8
    );
}

async function stopCamera() {

    if (window.localStream) {

        window.localStream
            .getTracks()
            .forEach(track => track.stop());

        window.localStream = null;
    }

    const video =
        document.getElementById('camera_video');

    if (video) {

        video.pause();
        video.srcObject = null;
    }

    const container =
        document.getElementById('camera_container');

    if (container) {
        container.remove();
    }

    return "Camera stopped";
}
'''))

# Start camera
eval_js('initCamera()')

def get_frame():
    # Call the JavaScript function to capture a frame
    data = eval_js('captureFrame()')

    # Decode the base64 image data
    binary = b64decode(data.split(',')[1])

    # Convert to numpy array
    jpg = np.frombuffer(binary, dtype=np.uint8)

    # Decode as OpenCV image
    frame = cv2.imdecode(jpg, cv2.IMREAD_COLOR)

    return frame


# Process frames
try:
    for _ in range(20):

        frame = get_frame()

        if frame is None:
            continue

        # Analyze face
        result = DeepFace.analyze(
            frame,
            actions=['emotion','gender'],
            enforce_detection=False
        )

        data = result[0]

        emotion = data['dominant_emotion']

        gender = data['dominant_gender']

        # Display text
        text = f"{gender}, {emotion}"

        cv2.putText(
            frame,
            text,
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2_imshow(frame)

        time.sleep(0.2)
finally:
      print(eval_js('stopCamera()'))

In [ ]:
from deepface import DeepFace
from ultralytics import YOLO

from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

from base64 import b64decode
import cv2
import numpy as np

# Load YOLO model
model = YOLO("yolov8n.pt")

# JavaScript camera capture
display(Javascript('''
async function takePhoto() {

    // Create container
    const div = document.createElement('div');
    div.id = 'camera_container';

    // Create button
    const capture = document.createElement('button');
    capture.textContent = 'Capture Photo';

    // Create video
    const video = document.createElement('video');
    video.id = 'camera_video';
    video.style.display = 'block';

    div.appendChild(video);
    div.appendChild(capture);

    document.body.appendChild(div);

    // Start webcam
    window.localStream =
        await navigator.mediaDevices.getUserMedia({
            video: true
        });

    video.srcObject = window.localStream;

    await video.play();

    // Wait for capture
    await new Promise(
        (resolve) => capture.onclick = resolve
    );

    // Create canvas
    const canvas =
        document.createElement('canvas');

    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    const ctx =
        canvas.getContext('2d');

    ctx.drawImage(video, 0, 0);

    // Convert image
    const data =
        canvas.toDataURL(
            'image/jpeg',
            0.8
        );

    // STOP CAMERA COMPLETELY
    video.pause();

    if (video.srcObject) {
        video.srcObject.getTracks().forEach(
            track => track.stop()
        );
    }

    video.srcObject = null;

    // Remove HTML elements
    div.remove();

    // Clear global stream
    window.localStream = null;

    return data;
}
'''))

# Capture image
data = eval_js('takePhoto()')

# Decode image
binary = b64decode(data.split(',')[1])

jpg = np.frombuffer(
    binary,
    dtype=np.uint8
)

frame = cv2.imdecode(
    jpg,
    cv2.IMREAD_COLOR
)

# Run YOLO detection
results = model(frame)

# Get detected boxes
boxes = results[0].boxes

# Process detections
for box in boxes:

    cls = int(box.cls[0])

    # Detect only person
    if model.names[cls] == 'person':

        x1, y1, x2, y2 = map(
            int,
            box.xyxy[0]
        )

        # Crop person
        person_crop = frame[y1:y2, x1:x2]

        # Skip invalid crop
        if person_crop.size == 0:
            continue

        try:

            # Analyze face
            analysis = DeepFace.analyze(
                person_crop,
                actions=['gender', 'emotion'],
                enforce_detection=False
            )

            result = analysis[0]

            gender = result['dominant_gender']
            emotion = result['dominant_emotion']

            label = f"{gender}, {emotion}"

        except:
            label = "Analysis Failed"

        # Draw rectangle
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Put label
        cv2.putText(
            frame,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

# Show final analyzed image
cv2_imshow(frame)

In [ ]:
from deepface import DeepFace
from ultralytics import YOLO

from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

from base64 import b64decode
import cv2
import numpy as np
import time

# Load YOLO model
model = YOLO("yolov8n.pt")

# Create webcam only once
display(Javascript('''
async function initCamera() {

    if (window.localStream) {
        return;
    }

    const div = document.createElement('div');
    div.id = 'camera_container';

    const video = document.createElement('video');
    video.id = 'camera_video';

    video.style.display = 'block';
    video.width = 400;

    document.body.appendChild(div);
    div.appendChild(video);

    window.localStream =
        await navigator.mediaDevices.getUserMedia({
            video: true
        });

    video.srcObject = window.localStream;

    await video.play();
}

async function captureFrame() {

    const video =
        document.getElementById('camera_video');

    const canvas =
        document.createElement('canvas');

    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    canvas
        .getContext('2d')
        .drawImage(video, 0, 0);

    return canvas.toDataURL(
        'image/jpeg',
        0.8
    );
}

async function stopCamera() {

    if (window.localStream) {

        window.localStream
            .getTracks()
            .forEach(track => track.stop());

        window.localStream = null;
    }

    const video =
        document.getElementById('camera_video');

    if (video) {

        video.pause();
        video.srcObject = null;
    }

    const container =
        document.getElementById('camera_container');

    if (container) {
        container.remove();
    }

    return "Camera stopped";
}
'''))

# Start camera
eval_js('initCamera()')

try:

    for _ in range(50):

        # Capture frame
        data = eval_js('captureFrame()')

        # Decode image
        binary = b64decode(data.split(',')[1])

        jpg = np.frombuffer(
            binary,
            dtype=np.uint8
        )

        frame = cv2.imdecode(
            jpg,
            cv2.IMREAD_COLOR
        )

        # YOLO detection
        results = model(frame)

        # Get detected boxes
        boxes = results[0].boxes

        # Process each detection
        for box in boxes:

            cls = int(box.cls[0])

            # Detect only persons
            if model.names[cls] == 'person':

                x1, y1, x2, y2 = map(
                    int,
                    box.xyxy[0]
                )

                # Crop detected person area
                person_crop = frame[y1:y2, x1:x2]

                # Skip empty crop
                if person_crop.size == 0:
                    continue

                try:

                    # Analyze gender and emotion
                    analysis = DeepFace.analyze(
                        person_crop,
                        actions=['gender', 'emotion'],
                        enforce_detection=False
                    )

                    result = analysis[0]

                    gender = result['dominant_gender']
                    emotion = result['dominant_emotion']

                    label = f"{gender}, {emotion}"

                except:
                    label = "Analysis Failed"

                # Draw rectangle
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )

                # Put text
                cv2.putText(
                    frame,
                    label,
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 0),
                    2
                )

        # Show final frame
        cv2_imshow(frame)

        time.sleep(0.1)

finally:

    print(eval_js('stopCamera()'))

In [ ]:
!pip install --upgrade nbformat
!pip install --upgrade nbconvert